In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

plt.style.use('ggplot')
plt.rcParams['image.cmap'] = 'viridis'

import matplotlib as mpl

viridis_colors = plt.cm.viridis(np.linspace(0, 1, 5))
plt.rcParams['axes.prop_cycle'] = mpl.cycler(color=viridis_colors)

## Rank analysis

In [ ]:
rank_output_folder_format = "../output/tau/rank/{total_ranks}/{sample}/profile.{rank}.0.0"
rank_range = range(1, 11)
num_samples = 3

time_data = []
event_data = []

for total_ranks in rank_range:
    for rank in range(total_ranks):
        for sample in range(1, num_samples + 1):
            with open(rank_output_folder_format.format(total_ranks=total_ranks, rank=rank, sample=sample)) as fd:
                line = fd.readline()
                section = "time"
                i = 0
                while line:
                    if i >= 3 and section == "time":
                        if "aggregates" in line:
                            i = 0
                            section = "event"
                            continue
                        _, name, rest, group, _ = line.split('"')
                        _, calls, subrs, excl, incl, p_calls, _ = rest.split(" ")
                        time_data.append([total_ranks, rank, sample, name, int(calls), int(subrs), float(excl)/1e6, float(incl)/1e6, int(p_calls), group])
                    if i >= 3 and section == "event":
                        _, name, rest = line.split('"')
                        _, numevents, max, min, mean, sumsqr = rest.split(" ")
                        event_data.append([total_ranks, rank, sample, name, int(numevents), float(max)/1e6, float(min)/1e6, float(mean)/1e6, float(sumsqr)/1e6])
                    line = fd.readline()
                    i += 1

time_df = pd.DataFrame(time_data, columns=["total_ranks", "rank", "sample", "function_name", "calls", "subrs", "excl", "incl", "p_calls", "group"])
event_df = pd.DataFrame(event_data, columns=["total_ranks", "rank", "sample", "event_name", "num_events", "max", "min", "mean", "sum_sqrt"])

In [ ]:
init_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::MisinformationDiffusionModel")].drop(["function_name", "group"], axis=1)
init_net_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::initNetwork__")].drop(["function_name", "group"], axis=1)
step_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::step()")].drop(["function_name", "group"], axis=1)
computation_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::step__stateCalc")].drop(["function_name", "group"], axis=1)
communication_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::step__ghost")].drop(["function_name", "group"], axis=1)
result_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::recordDynamicResults()")].drop(["function_name", "group"], axis=1)
total_time_df = time_df[time_df.function_name.str.startswith("main")].drop(["function_name", "group"], axis=1)
mpi_sync_time_df = time_df[time_df.function_name.str.startswith("MPI Coll")].drop(["function_name", "group"], axis=1)

In [ ]:
total_time = total_time_df.groupby(["total_ranks", "rank", "sample"]).mean().incl
total_time.name = "total_time"
init_time = init_time_df.groupby(["total_ranks", "rank", "sample"]).mean().incl
init_time.name = "initialization_time"
computation_time = computation_time_df.groupby(["total_ranks", "rank", "sample"]).mean().incl
computation_time.name = "computation_time"
communication_time = communication_time_df.groupby(["total_ranks", "rank", "sample"]).mean().incl
communication_time.name = "communication_time"
results_time = result_time_df.groupby(["total_ranks", "rank", "sample"]).mean().incl
results_time.name = "results_time"
mpi_sync_time = mpi_sync_time_df.groupby(["total_ranks", "rank", "sample"]).mean().incl
mpi_sync_time.name = "mpi_sync_time"

ranks_df = pd.concat(
    [total_time, init_time, computation_time, communication_time, results_time, mpi_sync_time],
    axis=1
)
ranks_df

In [ ]:
data = ranks_df.groupby("total_ranks").mean().drop(["total_time", "initialization_time"], axis=1)

fig, ax = plt.subplots(figsize=(16,9))
data.plot(kind="bar", stacked=True, ax=ax, colormap="viridis", edgecolor="#fefefe")
plt.xlabel("Ranks")
ax.set_xticklabels(data.index, rotation=0)
plt.ylabel("Time (seconds)")
plt.legend()
plt.show()

In [ ]:
for col in ranks_df.columns:
    plt.figure(figsize=(12,6))
    ranked_data = [ranks_df.loc[total_ranks][col].to_numpy() for total_ranks in rank_range]
    plt.boxplot(ranked_data, label=col)
    plt.xlabel("Ranks")
    plt.ylabel("Time (seconds)")
    plt.title(col.replace("_", " ").capitalize())
    plt.savefig(f"../figures/rank_vs_{col}_distribution.png")
    plt.show()


In [ ]:
for total_ranks in rank_range:
    init_time = init_time_df[init_time_df.total_ranks == total_ranks]
    total_time = total_time_df[total_time_df.total_ranks == total_ranks]
    computation_time = computation_time_df[computation_time_df.total_ranks == total_ranks]
    communication_time = communication_time_df[communication_time_df.total_ranks == total_ranks]
    result_time = result_time_df[result_time_df.total_ranks == total_ranks]
    mpi_sync_time = mpi_sync_time_df[mpi_sync_time_df.total_ranks == total_ranks]

    initialization_ratio = init_time.incl.mean() / total_time.incl.mean()
    computation_ratio = computation_time.incl.mean() / total_time.incl.mean()
    communication_ratio = communication_time.incl.mean() / total_time.incl.mean()
    results_output_ratio = result_time.incl.mean() / total_time.incl.mean()

    print(f"Total ranks: {total_ranks}")
    print(f"- Total time per rank: {', '.join([f'{t:.3f} sec' for t in total_time.incl])}")
    print(f"- Mean total time: {total_time.incl.mean():.3f}")
    print(f"- Load imbalance per rank: {', '.join([f'{a/b*100:.2f} %' for a, b in zip(mpi_sync_time.incl, total_time.incl)])}")
    print(f"- Mean load imbalance: {sum([a/b*100 for a, b in zip(mpi_sync_time.incl, total_time.incl)]) / len(mpi_sync_time):.2f} %")
    print(f"- Initialization Ratio: {initialization_ratio * 100:.2f} %")
    print(f"- Computation Ratio: {computation_ratio * 100:.2f} %")
    print(f"- Communication Ratio: {communication_ratio * 100:.2f} %")
    print(f"- Writing Ratio: {results_output_ratio * 100:.2f} %")

## Agent number analysis

In [ ]:
size_output_folder_format = "../output/tau/network_size/{size}/{sample}/profile.{rank}.0.0"
#sizes = [500, 1000, 5000, 10000, 15000, 20000, 50000]
sizes = [1000, 10000, 20000, 30000, 40000, 50000]

total_ranks = 4
num_samples = 3

time_data = []
event_data = []

for size in sizes:
    for rank in range(total_ranks):
        for sample in range(1, num_samples + 1):
            with open(size_output_folder_format.format(size=size, rank=rank, sample=sample)) as fd:
                line = fd.readline()
                section = "time"
                i = 0
                while line:
                    if i >= 3 and section == "time":
                        if "aggregates" in line:
                            i = 0
                            section = "event"
                            continue
                        _, name, rest, group, _ = line.split('"')
                        _, calls, subrs, excl, incl, p_calls, _ = rest.split(" ")
                        time_data.append([size, rank, sample, name, int(calls), int(subrs), float(excl)/1e6, float(incl)/1e6, int(p_calls), group])
                    if i >= 3 and section == "event":
                        _, name, rest = line.split('"')
                        _, numevents, max, min, mean, sumsqr = rest.split(" ")
                        event_data.append([size, rank, sample, name, int(numevents), float(max)/1e6, float(min)/1e6, float(mean)/1e6, float(sumsqr)/1e6])
                    line = fd.readline()
                    i += 1

time_df = pd.DataFrame(time_data, columns=["size", "rank", "sample", "function_name", "calls", "subrs", "excl", "incl", "p_calls", "group"])
event_df = pd.DataFrame(event_data, columns=["size", "rank", "sample", "event_name", "num_events", "max", "min", "mean", "sum_sqrt"])

init_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::MisinformationDiffusionModel")].drop(["function_name", "group"], axis=1)
init_net_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::initNetwork__")].drop(["function_name", "group"], axis=1)
step_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::step()")].drop(["function_name", "group"], axis=1)
computation_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::step__stateCalc")].drop(["function_name", "group"], axis=1)
communication_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::step__ghost")].drop(["function_name", "group"], axis=1)
result_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::recordDynamicResults()")].drop(["function_name", "group"], axis=1)
total_time_df = time_df[time_df.function_name.str.startswith("main")].drop(["function_name", "group"], axis=1)
mpi_sync_time_df = time_df[time_df.function_name.str.startswith("MPI Coll")].drop(["function_name", "group"], axis=1)

In [ ]:
total_time = total_time_df.groupby(["size", "rank", "sample"]).mean().incl
total_time.name = "total_time"
init_time = init_time_df.groupby(["size", "rank", "sample"]).mean().incl
init_time.name = "initialization_time"
computation_time = computation_time_df.groupby(["size", "rank", "sample"]).mean().incl
computation_time.name = "computation_time"
communication_time = communication_time_df.groupby(["size", "rank", "sample"]).mean().incl
communication_time.name = "communication_time"
results_time = result_time_df.groupby(["size", "rank", "sample"]).mean().incl
results_time.name = "results_time"
mpi_sync_time = mpi_sync_time_df.groupby(["size", "rank", "sample"]).mean().incl
mpi_sync_time.name = "mpi_sync_time"

sizes_df = pd.concat(
    [total_time, init_time, computation_time, communication_time, results_time, mpi_sync_time],
    axis=1
)
sizes_df

In [ ]:
data = sizes_df.groupby("size").mean().drop(["total_time", "initialization_time"], axis=1)

fig, ax = plt.subplots(figsize=(16,9))
data.plot(kind="area", stacked=True, ax=ax, alpha=0.75, linewidth=0)
data.plot(kind="line", stacked=True, ax=ax, color="#fefefe", linewidth=0.5, legend=False)
plt.xlabel("Agent count")
plt.ylabel("Time (seconds)")
plt.show()

In [ ]:
total_time = sizes_df.computation_time + sizes_df.communication_time + sizes_df.results_time + sizes_df.mpi_sync_time
sizes_df["computation_ratio"] = sizes_df.computation_time / total_time
sizes_df["communication_ratio"] = sizes_df.communication_time / total_time
sizes_df["results_ratio"] = sizes_df.results_time / total_time
sizes_df["mpi_sync_ratio"] = sizes_df.mpi_sync_time / total_time

data = sizes_df.groupby("size").mean().drop(["total_time", "initialization_time", "computation_time", "communication_time", "results_time", "mpi_sync_time" ], axis=1)

fig, ax = plt.subplots(figsize=(16,9))
data.plot(kind="area", stacked=True, ax=ax, alpha=0.75, linewidth=0)
data.plot(kind="line", stacked=True, ax=ax, color="#fefefe", linewidth=0.5, legend=False)
plt.xlabel("Agent count")
plt.ylabel("Total time ratio")
plt.show()

## Partition strategy analysis

In [ ]:
partition_output_folder_format = "../output/tau/partition/{strat}/{sample}/profile.{rank}.0.0"
strats = ["node_id_modulo", "node_community_modulo"]
total_ranks = 4
num_samples = 3

time_data = []
event_data = []

for strat in strats:
    for rank in range(total_ranks):
        for sample in range(1, num_samples + 1):
            with open(partition_output_folder_format.format(strat=strat, rank=rank, sample=sample)) as fd:
                line = fd.readline()
                section = "time"
                i = 0
                while line:
                    if i >= 3 and section == "time":
                        if "aggregates" in line:
                            i = 0
                            section = "event"
                            continue
                        _, name, rest, group, _ = line.split('"')
                        _, calls, subrs, excl, incl, p_calls, _ = rest.split(" ")
                        time_data.append([strat, rank, sample, name, int(calls), int(subrs), float(excl)/1e6, float(incl)/1e6, int(p_calls), group])
                    if i >= 3 and section == "event":
                        _, name, rest = line.split('"')
                        _, numevents, max, min, mean, sumsqr = rest.split(" ")
                        event_data.append([strat, rank, sample, name, int(numevents), float(max)/1e6, float(min)/1e6, float(mean)/1e6, float(sumsqr)/1e6])
                    line = fd.readline()
                    i += 1

time_df = pd.DataFrame(time_data, columns=["partitioning_strat", "rank", "sample", "function_name", "calls", "subrs", "excl", "incl", "p_calls", "group"])
event_df = pd.DataFrame(event_data, columns=["partitioning_strat", "rank", "sample", "event_name", "num_events", "max", "min", "mean", "sum_sqrt"])

init_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::MisinformationDiffusionModel")].drop(["function_name", "group"], axis=1)
init_net_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::initNetwork__")].drop(["function_name", "group"], axis=1)
step_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::step()")].drop(["function_name", "group"], axis=1)
computation_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::step__stateCalc")].drop(["function_name", "group"], axis=1)
communication_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::step__ghost")].drop(["function_name", "group"], axis=1)
result_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::recordDynamicResults()")].drop(["function_name", "group"], axis=1)
total_time_df = time_df[time_df.function_name.str.startswith("main")].drop(["function_name", "group"], axis=1)
mpi_sync_time_df = time_df[time_df.function_name.str.startswith("MPI Coll")].drop(["function_name", "group"], axis=1)

message_size_rx_df = event_df[event_df.event_name.str.startswith("Message size received")].drop(["event_name"], axis=1)
message_size_tx_df = event_df[event_df.event_name.str.startswith("Message size sent")].drop(["event_name"], axis=1)

In [ ]:
message_size_rx = message_size_rx_df.groupby(["partitioning_strat", "rank", "sample"]).mean()["mean"]
message_size_rx.name = "message_size_rx"
message_size_tx = message_size_tx_df.groupby(["partitioning_strat", "rank", "sample"]).mean()["mean"]
message_size_tx.name = "message_size_tx"
messages_received = message_size_rx_df.groupby(["partitioning_strat", "rank", "sample"]).mean()["num_events"]
messages_received.name = "messages_received"
messages_sent = message_size_tx_df.groupby(["partitioning_strat", "rank", "sample"]).mean()["num_events"]
messages_sent.name = "messages_sent"

strats_df = pd.concat(
    [message_size_rx, message_size_tx, messages_received, messages_sent],
    axis=1
)
strats_df

In [ ]:
data = strats_df.groupby("partitioning_strat").mean().drop(["messages_received", "messages_sent"], axis=1)
data.index = ["Community ID", "Node ID"]
data.columns = ["Received", "Sent"]

fig, ax = plt.subplots(figsize=(6,9))
data.plot(kind="bar", stacked=True, ax=ax, edgecolor="#fefefe")
plt.xlabel("Partitioning strategy")
ax.set_xticklabels(data.index, rotation=0)
plt.ylabel("MPI message size")
plt.legend()
plt.show()

In [ ]:
total_time = total_time_df.groupby(["partitioning_strat", "rank", "sample"]).mean().incl
total_time.name = "total_time"
init_time = init_time_df.groupby(["partitioning_strat", "rank", "sample"]).mean().incl
init_time.name = "initialization_time"
computation_time = computation_time_df.groupby(["partitioning_strat", "rank", "sample"]).mean().incl
computation_time.name = "computation_time"
communication_time = communication_time_df.groupby(["partitioning_strat", "rank", "sample"]).mean().incl
communication_time.name = "communication_time"
results_time = result_time_df.groupby(["partitioning_strat", "rank", "sample"]).mean().incl
results_time.name = "results_time"
mpi_sync_time = mpi_sync_time_df.groupby(["partitioning_strat", "rank", "sample"]).mean().incl
mpi_sync_time.name = "mpi_sync_time"

strats_df = pd.concat(
    [total_time, init_time, computation_time, communication_time, results_time, mpi_sync_time],
    axis=1
)
strats_df

In [ ]:
data = strats_df.groupby("partitioning_strat").mean().drop(["total_time"], axis=1)
data.index = ["Community ID", "Node ID"]
data.columns = ["Initialization", "Computation", "Communication", "Results", "MPI Sync"]

fig, ax = plt.subplots(figsize=(6,9))
data.plot(kind="bar", stacked=True, ax=ax, edgecolor="#fefefe")
plt.xlabel("Partitioning strategy")
ax.set_xticklabels(data.index, rotation=0)
plt.ylabel("Time (seconds)")
plt.ylim((0, 16))
plt.legend()
plt.show()

## Rank and Agent count analysis

In [ ]:
output_folder_format = "../output/tau/combined/rank_{total_ranks}/size_{size}/{sample}/profile.{rank}.0.0"
strats = ["node_id_modulo", "node_community_modulo"]
rank_range = range(1, 11)
sizes = [1000, 10000, 20000, 30000, 40000, 50000]
num_samples = 3

time_data = []
event_data = []

for total_ranks in rank_range:
    for rank in range(total_ranks):
        for size in sizes:
            for sample in range(1, num_samples + 1):
                with open(output_folder_format.format(total_ranks=total_ranks, rank=rank, size=size, sample=sample)) as fd:
                    line = fd.readline()
                    section = "time"
                    i = 0
                    while line:
                        if i >= 3 and section == "time":
                            if "aggregates" in line:
                                i = 0
                                section = "event"
                                continue
                            _, name, rest, group, _ = line.split('"')
                            _, calls, subrs, excl, incl, p_calls, _ = rest.split(" ")
                            time_data.append([total_ranks, size, rank, sample, name, int(calls), int(subrs), float(excl)/1e6, float(incl)/1e6, int(p_calls), group])
                        if i >= 3 and section == "event":
                            _, name, rest = line.split('"')
                            _, numevents, max, min, mean, sumsqr = rest.split(" ")
                            event_data.append([total_ranks, size, rank, sample, name, int(numevents), float(max)/1e6, float(min)/1e6, float(mean)/1e6, float(sumsqr)/1e6])
                        line = fd.readline()
                        i += 1

time_df = pd.DataFrame(time_data, columns=["total_ranks", "size", "rank", "sample", "function_name", "calls", "subrs", "excl", "incl", "p_calls", "group"])
event_df = pd.DataFrame(event_data, columns=["total_ranks", "size", "rank", "sample", "event_name", "num_events", "max", "min", "mean", "sum_sqrt"])

init_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::MisinformationDiffusionModel")].drop(["function_name", "group"], axis=1)
init_net_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::initNetwork__")].drop(["function_name", "group"], axis=1)
step_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::step()")].drop(["function_name", "group"], axis=1)
computation_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::step__stateCalc")].drop(["function_name", "group"], axis=1)
communication_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::step__ghost")].drop(["function_name", "group"], axis=1)
result_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::recordDynamicResults()")].drop(["function_name", "group"], axis=1)
total_time_df = time_df[time_df.function_name.str.startswith("main")].drop(["function_name", "group"], axis=1)
mpi_sync_time_df = time_df[time_df.function_name.str.startswith("MPI Coll")].drop(["function_name", "group"], axis=1)

In [ ]:
total_time = total_time_df.groupby(["total_ranks", "size", "rank", "sample"]).mean().incl
total_time.name = "total_time"
init_time = init_time_df.groupby(["total_ranks", "size", "rank", "sample"]).mean().incl
init_time.name = "initialization_time"
computation_time = computation_time_df.groupby(["total_ranks", "size", "rank", "sample"]).mean().incl
computation_time.name = "computation_time"
communication_time = communication_time_df.groupby(["total_ranks", "size", "rank", "sample"]).mean().incl
communication_time.name = "communication_time"
results_time = result_time_df.groupby(["total_ranks", "size", "rank", "sample"]).mean().incl
results_time.name = "results_time"
mpi_sync_time = mpi_sync_time_df.groupby(["total_ranks", "size", "rank", "sample"]).mean().incl
mpi_sync_time.name = "mpi_sync_time"

combined_df = pd.concat(
    [total_time, init_time, computation_time, communication_time, results_time, mpi_sync_time],
    axis=1
)
total_time_by_rank_and_size = combined_df.groupby(["total_ranks", "size"]).mean()
print(total_time_by_rank_and_size['total_time'] - total_time_by_rank_and_size["initialization_time"])
df_pivot = (total_time_by_rank_and_size['total_time'] - total_time_by_rank_and_size["initialization_time"]).unstack(level='size')

X = df_pivot.index.values        # Represents the unique values of your first index
Y = df_pivot.columns.values      # Represents the unique values of your second index
Z = df_pivot.T.values              # The 2D numpy array of your target metrics

fig, ax = plt.subplots(figsize=(15, 12))
contour = ax.contourf(X, Y, Z, levels=10, alpha = 0.75)
contour_lines = ax.contour(X, Y, Z, levels=10, colors="#fefefe", linewidths=0.75)
plt.colorbar(contour, label='Time (seconds)')
ax.clabel(contour_lines, fontsize=10)
plt.xlabel('Total ranks')
plt.ylabel('Agent count')

plt.show()

# Network structure analysis

In [ ]:
struct_output_folder_format = "../output/tau/structure/{structure}/{sample}/profile.{rank}.0.0"
# structures = ["ws__k_10_p_0.05", "ws__k_10_p_0.5", "ws__k_100_p_0.01", "ws__k_100_p_0.05", "ws__k_100_p_0.08", "ws__k_100_p_0.1", "ws__k_100_p_0.5", "ws__k_100_p_0.8"]
# structures += ["ba__m_2", "ba__m_10"]
structures = [
      "hk__m_2_p_0.9",
  "hk__m_10_p_0.9",
  "hk__m_5_p_0.9",
  "hk__m_2_p_0.3",
  "hk__m_10_p_0.3",
  "hk__m_5_p_0.3",
  "hk__m_2_p_0.1",
  "hk__m_10_p_0.1",
  "hk__m_5_p_0.1",
  "ws__k_40_p_0.01",
  "ws__k_40_p_0.2",
  "ws__k_40_p_0.8",
  "ws__k_60_p_0.01",
  "ws__k_60_p_0.2",
  "ws__k_60_p_0.8",
  "ws__k_80_p_0.01",
  "ws__k_80_p_0.2",
  "ws__k_80_p_0.8",
  "ws__k_100_p_0.01",
  "ws__k_100_p_0.1",
  "ws__k_100_p_0.05",
  "ws__k_100_p_0.2",
  "ws__k_100_p_0.5",
  "ws__k_100_p_0.8",
]
total_ranks = 4
num_samples = 1

time_data = []
event_data = []

for struct in structures:
    for rank in range(total_ranks):
        for sample in range(1, num_samples + 1):
            with open(struct_output_folder_format.format(structure=struct, rank=rank, sample=sample)) as fd:
                line = fd.readline()
                section = "time"
                i = 0
                while line:
                    if i >= 3 and section == "time":
                        if "aggregates" in line:
                            i = 0
                            section = "event"
                            continue
                        _, name, rest, group, _ = line.split('"')
                        _, calls, subrs, excl, incl, p_calls, _ = rest.split(" ")
                        time_data.append([struct, rank, sample, name, int(calls), int(subrs), float(excl)/1e6, float(incl)/1e6, int(p_calls), group])
                    if i >= 3 and section == "event":
                        _, name, rest = line.split('"')
                        _, numevents, max, min, mean, sumsqr = rest.split(" ")
                        event_data.append([struct, rank, sample, name, int(numevents), float(max)/1e6, float(min)/1e6, float(mean)/1e6, float(sumsqr)/1e6])
                    line = fd.readline()
                    i += 1

time_df = pd.DataFrame(time_data, columns=["network_struct", "rank", "sample", "function_name", "calls", "subrs", "excl", "incl", "p_calls", "group"])
event_df = pd.DataFrame(event_data, columns=["network_struct", "rank", "sample", "event_name", "num_events", "max", "min", "mean", "sum_sqrt"])

init_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::MisinformationDiffusionModel")].drop(["function_name", "group"], axis=1)
init_net_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::initNetwork__")].drop(["function_name", "group"], axis=1)
step_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::step()")].drop(["function_name", "group"], axis=1)
computation_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::step__stateCalc")].drop(["function_name", "group"], axis=1)
communication_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::step__ghost")].drop(["function_name", "group"], axis=1)
result_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::recordDynamicResults()")].drop(["function_name", "group"], axis=1)
total_time_df = time_df[time_df.function_name.str.startswith("main")].drop(["function_name", "group"], axis=1)
mpi_sync_time_df = time_df[time_df.function_name.str.startswith("MPI Coll")].drop(["function_name", "group"], axis=1)

In [ ]:
total_time = total_time_df.groupby(["network_struct", "rank", "sample"]).mean().incl
total_time.name = "total_time"
init_time = init_time_df.groupby(["network_struct", "rank", "sample"]).mean().incl
init_time.name = "initialization_time"
computation_time = computation_time_df.groupby(["network_struct", "rank", "sample"]).mean().incl
computation_time.name = "computation_time"
communication_time = communication_time_df.groupby(["network_struct", "rank", "sample"]).mean().incl
communication_time.name = "communication_time"
results_time = result_time_df.groupby(["network_struct", "rank", "sample"]).mean().incl
results_time.name = "results_time"
mpi_sync_time = mpi_sync_time_df.groupby(["network_struct", "rank", "sample"]).mean().incl
mpi_sync_time.name = "mpi_sync_time"

structs_df = pd.concat(
    [total_time, init_time, computation_time, communication_time, results_time, mpi_sync_time],
    axis=1
)
structs_df

In [ ]:
data = structs_df.groupby("network_struct").mean().drop(["total_time", "initialization_time"], axis=1)

fig, ax = plt.subplots(figsize=(16,9))
data.plot(kind="bar", stacked=True, ax=ax, edgecolor="#fefefe")
plt.xlabel("Partitioning strategy")
plt.ylabel("Time (seconds)")
plt.legend()
plt.show()

In [ ]:
data = structs_df.groupby("network_struct").mean()

# X = []
# Y = []
# Z = []
# for network in data.index:
#     G = nx.read_gml(f"../props/networks/{network}_network_1000.gml")
#     X.append(sum(dict(G.degree()).values()) / G.number_of_nodes())
#     Y.append(nx.average_clustering(G))
#     Z.append(data.loc[network].total_time - data.loc[network].initialization_time)

fig, ax = plt.subplots(figsize=(16, 12))
sc = ax.scatter(X, Y, c=Z, edgecolors="white", s=200)
plt.colorbar(sc, label='Time (seconds)')
plt.xlabel('Avg Degree')
plt.ylabel('Avg Clustering')


for i, txt in enumerate(data.index):
    ax.annotate(txt, (X[i]-4.6, Y[i]+0.015), color="#666", size=8)

fig, ax = plt.subplots(figsize=(16, 12))
ax.tricontour(X, Y, Z, levels=14, linewidths=0.5, colors='white')
cntr = ax.tricontourf(X, Y, Z, levels=14, alpha=0.75)

plt.colorbar(cntr, label='Time (seconds)')
plt.xlabel('Avg Degree')
plt.ylabel('Avg Clustering')
ax.plot(X, Y, 'ko', ms=3)

## Extra compute analysis

In [ ]:
cycles_output_folder_format = "../output/tau/compute_cycles/{cycles}_cycles/{sample}/profile.{rank}.0.0"
cycles = [0, 1000, 5000, 10000]

total_ranks = 4
num_samples = 3

time_data = []
event_data = []

for cycle in cycles:
    for rank in range(total_ranks):
        for sample in range(1, num_samples + 1):
            with open(cycles_output_folder_format.format(cycles=cycle, rank=rank, sample=sample)) as fd:
                line = fd.readline()
                section = "time"
                i = 0
                while line:
                    if i >= 3 and section == "time":
                        if "aggregates" in line:
                            i = 0
                            section = "event"
                            continue
                        _, name, rest, group, _ = line.split('"')
                        _, calls, subrs, excl, incl, p_calls, _ = rest.split(" ")
                        time_data.append([cycle, rank, sample, name, int(calls), int(subrs), float(excl)/1e6, float(incl)/1e6, int(p_calls), group])
                    if i >= 3 and section == "event":
                        _, name, rest = line.split('"')
                        _, numevents, max, min, mean, sumsqr = rest.split(" ")
                        event_data.append([cycle, rank, sample, name, int(numevents), float(max)/1e6, float(min)/1e6, float(mean)/1e6, float(sumsqr)/1e6])
                    line = fd.readline()
                    i += 1

time_df = pd.DataFrame(time_data, columns=["extra_computation_cycles", "rank", "sample", "function_name", "calls", "subrs", "excl", "incl", "p_calls", "group"])
event_df = pd.DataFrame(event_data, columns=["extra_computation_cycles", "rank", "sample", "event_name", "num_events", "max", "min", "mean", "sum_sqrt"])

init_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::MisinformationDiffusionModel")].drop(["function_name", "group"], axis=1)
init_net_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::initNetwork__")].drop(["function_name", "group"], axis=1)
step_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::step()")].drop(["function_name", "group"], axis=1)
computation_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::step__stateCalc")].drop(["function_name", "group"], axis=1)
communication_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::step__ghost")].drop(["function_name", "group"], axis=1)
result_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::recordDynamicResults()")].drop(["function_name", "group"], axis=1)
total_time_df = time_df[time_df.function_name.str.startswith("main")].drop(["function_name", "group"], axis=1)
mpi_sync_time_df = time_df[time_df.function_name.str.startswith("MPI Coll")].drop(["function_name", "group"], axis=1)

total_time = total_time_df.groupby(["extra_computation_cycles", "rank", "sample"]).mean().incl
total_time.name = "total_time"
init_time = init_time_df.groupby(["extra_computation_cycles", "rank", "sample"]).mean().incl
init_time.name = "initialization_time"
computation_time = computation_time_df.groupby(["extra_computation_cycles", "rank", "sample"]).mean().incl
computation_time.name = "computation_time"
communication_time = communication_time_df.groupby(["extra_computation_cycles", "rank", "sample"]).mean().incl
communication_time.name = "communication_time"
results_time = result_time_df.groupby(["extra_computation_cycles", "rank", "sample"]).mean().incl
results_time.name = "results_time"
mpi_sync_time = mpi_sync_time_df.groupby(["extra_computation_cycles", "rank", "sample"]).mean().incl
mpi_sync_time.name = "mpi_sync_time"

cycles_df = pd.concat(
    [total_time, init_time, computation_time, communication_time, results_time, mpi_sync_time],
    axis=1
)
cycles_df

In [ ]:
data = cycles_df.groupby("extra_computation_cycles").mean().drop(["total_time", "initialization_time"], axis=1)

fig, ax = plt.subplots(figsize=(16,9))
data.plot(kind="area", stacked=True, ax=ax, alpha=0.75, linewidth=0)
data.plot(kind="line", stacked=True, ax=ax, color="#fefefe", linewidth=0.5, legend=False)
plt.xlabel("Extra computation cycles")
plt.ylabel("Time (seconds)")
plt.show()

In [ ]:
total_time = cycles_df.computation_time + cycles_df.communication_time + cycles_df.results_time + cycles_df.mpi_sync_time
cycles_df["computation_ratio"] = cycles_df.computation_time / total_time
cycles_df["communication_ratio"] = cycles_df.communication_time / total_time
cycles_df["results_ratio"] = cycles_df.results_time / total_time
cycles_df["mpi_sync_ratio"] = cycles_df.mpi_sync_time / total_time

data = cycles_df.groupby("extra_computation_cycles").mean().drop(["total_time", "initialization_time", "computation_time", "communication_time", "results_time", "mpi_sync_time" ], axis=1)

fig, ax = plt.subplots(figsize=(15,9))
data.plot(kind="area", stacked=True, ax=ax, alpha=0.75, linewidth=0)
data.plot(kind="line", stacked=True, ax=ax, color="#fefefe", linewidth=0.5, legend=False)
plt.xlabel("Extra compute cycles")
plt.ylabel("Total time ratio")
plt.xlim((0,10000))
plt.ylim((0,1))
plt.show()

In [ ]:
message_output_folder_format = "../output/tau/message_size/{message_size}_bytes/{sample}/profile.{rank}.0.0"
message_sizes = [0, 1024, 10240, 20480]

total_ranks = 4
num_samples = 3

time_data = []
event_data = []

for message_size in message_sizes:
    for rank in range(total_ranks):
        for sample in range(1, num_samples + 1):
            with open(message_output_folder_format.format(message_size=message_size, rank=rank, sample=sample)) as fd:
                line = fd.readline()
                section = "time"
                i = 0
                while line:
                    if i >= 3 and section == "time":
                        if "aggregates" in line:
                            i = 0
                            section = "event"
                            continue
                        _, name, rest, group, _ = line.split('"')
                        _, calls, subrs, excl, incl, p_calls, _ = rest.split(" ")
                        time_data.append([message_size, rank, sample, name, int(calls), int(subrs), float(excl)/1e6, float(incl)/1e6, int(p_calls), group])
                    if i >= 3 and section == "event":
                        _, name, rest = line.split('"')
                        _, numevents, max, min, mean, sumsqr = rest.split(" ")
                        event_data.append([message_size, rank, sample, name, int(numevents), float(max)/1e6, float(min)/1e6, float(mean)/1e6, float(sumsqr)/1e6])
                    line = fd.readline()
                    i += 1

time_df = pd.DataFrame(time_data, columns=["extra_message_size", "rank", "sample", "function_name", "calls", "subrs", "excl", "incl", "p_calls", "group"])
event_df = pd.DataFrame(event_data, columns=["extra_message_size", "rank", "sample", "event_name", "num_events", "max", "min", "mean", "sum_sqrt"])

init_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::MisinformationDiffusionModel")].drop(["function_name", "group"], axis=1)
init_net_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::initNetwork__")].drop(["function_name", "group"], axis=1)
step_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::step()")].drop(["function_name", "group"], axis=1)
computation_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::step__stateCalc")].drop(["function_name", "group"], axis=1)
communication_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::step__ghost")].drop(["function_name", "group"], axis=1)
result_time_df = time_df[time_df.function_name.str.startswith("MisinformationDiffusionModel::recordDynamicResults()")].drop(["function_name", "group"], axis=1)
total_time_df = time_df[time_df.function_name.str.startswith("main")].drop(["function_name", "group"], axis=1)
mpi_sync_time_df = time_df[time_df.function_name.str.startswith("MPI Coll")].drop(["function_name", "group"], axis=1)

total_time = total_time_df.groupby(["extra_message_size", "rank", "sample"]).mean().incl
total_time.name = "total_time"
init_time = init_time_df.groupby(["extra_message_size", "rank", "sample"]).mean().incl
init_time.name = "initialization_time"
computation_time = computation_time_df.groupby(["extra_message_size", "rank", "sample"]).mean().incl
computation_time.name = "computation_time"
communication_time = communication_time_df.groupby(["extra_message_size", "rank", "sample"]).mean().incl
communication_time.name = "communication_time"
results_time = result_time_df.groupby(["extra_message_size", "rank", "sample"]).mean().incl
results_time.name = "results_time"
mpi_sync_time = mpi_sync_time_df.groupby(["extra_message_size", "rank", "sample"]).mean().incl
mpi_sync_time.name = "mpi_sync_time"

message_size_df = pd.concat(
    [total_time, init_time, computation_time, communication_time, results_time, mpi_sync_time],
    axis=1
)
message_size_df

In [ ]:
data = message_size_df.groupby("extra_message_size").mean().drop(["total_time", "initialization_time"], axis=1)

fig, ax = plt.subplots(figsize=(16,9))
data.plot(kind="area", stacked=True, ax=ax, alpha=0.75, linewidth=0)
data.plot(kind="line", stacked=True, ax=ax, color="#fefefe", linewidth=0.5, legend=False)
plt.xlabel("Extra message size (bytes)")
plt.ylabel("Time (seconds)")
plt.show()

In [ ]:
total_time = message_size_df.computation_time + message_size_df.communication_time + message_size_df.results_time + message_size_df.mpi_sync_time
message_size_df["computation_ratio"] = message_size_df.computation_time / total_time
message_size_df["communication_ratio"] = message_size_df.communication_time / total_time
message_size_df["results_ratio"] = message_size_df.results_time / total_time
message_size_df["mpi_sync_ratio"] = message_size_df.mpi_sync_time / total_time

data = message_size_df.groupby("extra_message_size").mean().drop(["total_time", "initialization_time", "computation_time", "communication_time", "results_time", "mpi_sync_time" ], axis=1)

fig, ax = plt.subplots(figsize=(15,9))
data.plot(kind="area", stacked=True, ax=ax, alpha=0.75, linewidth=0)
data.plot(kind="line", stacked=True, ax=ax, color="#fefefe", linewidth=0.5, legend=False)
plt.xlabel("Extra message size (bytes)")
plt.ylabel("Total time ratio")
plt.xlim((0,40960/2))
plt.ylim((0,1))
plt.show()